# STEP 1 — Import Libraries

In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split
# import transformers
# print(transformers.__version__)
# import torch
# print(torch.__version__)
# print(torch.version.cuda)


from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

# STEP 2 — Load Dataset (final_combined.csv)

In [26]:
df = pd.read_csv("../../datasets/processed/final_combined.csv")

# Keep only subject + label
df_subject = df[["subject", "label"]].dropna()
df_subject["subject"] = df_subject["subject"].astype(str)

print(df_subject.head())
print("Total samples:", len(df_subject))

                                        subject  label
0  new Catholic mailing list now up and running      0
1                                       re[12]:      1
2                Take a moment to explore this.      1
3                                     Greetings      0
4                       LOANS @ 3.17% (27 term)      1
Total samples: 120973


/tmp/ipykernel_722100/3736549266.py:1: DtypeWarning: Columns (3,4,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../datasets/processed/final_combined.csv")


# STEP 3 — Train/Validation Split

In [27]:
train_df, val_df = train_test_split(
    df_subject,
    test_size=0.1,
    stratify=df_subject["label"],
    random_state=42
)

print(len(train_df), len(val_df))


108875 12098


# STEP 4 — Build Tokenizer

In [28]:
model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

/home/defalt/.pyenv/versions/venv-3.10.12/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


# STEP 5 — HuggingFace Dataset Format

In [29]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)

def tokenize(batch):
    return tokenizer(
        batch["subject"],
        truncation=True,
        max_length=64  # subject lines are short
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)

train_ds = train_ds.remove_columns(["subject", "__index_level_0__"])
val_ds   = val_ds.remove_columns(["subject", "__index_level_0__"])


Map: 100%|██████████| 12098/12098 [00:00<00:00, 77263.44 examples/s]


# STEP 6 — Load Model (DeBERTa-v3-base)

In [30]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# Enable gradient checkpointing for VRAM efficiency
# model.gradient_checkpointing_enable()


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# STEP 7 — Training Arguments

In [31]:
training_args = TrainingArguments(
    output_dir="subject_model_out",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,                          # Mixed Precision
    logging_steps=100,
    report_to="none",
)


# STEP 8 — Data Collator

In [32]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# STEP 9 — Trainer

In [33]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator
)


/tmp/ipykernel_722100/2941208116.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# STEP 10 — Train

In [34]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

# STEP 11 — Save Model + Tokenizer

In [ ]:
trainer.save_model("subject_encoder") 
tokenizer.save_pretrained("subject_encoder_tokenizer")